# 13 — Last.fm Popularity Feature

Builds `album_lastfm_popularity_matrix.npz` from the scraped Last.fm parquet.

**Input:** `data/lastfm_data.parquet` (produced by merging all worker parquets in `lastfm_scraper.ipynb`)

**Output:** `data/features/album_lastfm_popularity_matrix.npz`
- 4 columns: `album_listeners`, `album_scrobbles`, `artist_listeners`, `artist_scrobbles`
- All min-max scaled to [0, 1]
- Same row order as `data/features/album_ids.pkl` (1,758,488 albums)
- Albums not in Last.fm data → all zeros (sparse)

In [ ]:
import os, re, pickle
import pandas as pd
import numpy as np
from scipy.sparse import lil_matrix, save_npz, load_npz

DATA_DIR     = '../data'
FEATURES_DIR = '../data/features'
INPUT_PATH   = os.path.join(DATA_DIR, 'lastfm_data.parquet')

## 1. Load the scraped parquet

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(f'Rows loaded: {len(df):,}')
print(df.dtypes)
df.head(3)

In [ ]:
# Rename to safe internal names
df = df.rename(columns={
    'Artist':           'artist_name',
    'Album':            'album_name',
    'Artist_Listeners': 'artist_listeners',
    'Artist_Scrobbles': 'artist_scrobbles',
    'Album_Listeners':  'album_listeners',
    'Album_Scrobbles':  'album_scrobbles',
    'Similar_Artists':  'similar_artists',
})

df = df[['artist_name', 'album_name',
         'album_listeners', 'album_scrobbles',
         'artist_listeners', 'artist_scrobbles']].copy()

# Convert numeric cols — handles commas ("1,234,567") and N/A values
for col in ['album_listeners', 'album_scrobbles', 'artist_listeners', 'artist_scrobbles']:
    df[col] = (
        df[col].astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
        .replace({'': np.nan, 'N/A': np.nan, 'None Found': np.nan})
        .astype(float)
        .fillna(0)
    )

print(f'After cleaning: {len(df):,} rows')
df.describe()

## 2. Normalise names & deduplicate

In [ ]:
def normalise(s):
    if not isinstance(s, str): return ''
    s = s.lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

df['artist_norm'] = df['artist_name'].map(normalise)
df['album_norm']  = df['album_name'].map(normalise)

# Keep highest-scrobbles row per (artist, album)
df = (
    df.sort_values('album_scrobbles', ascending=False)
      .drop_duplicates(subset=['artist_norm', 'album_norm'])
      .reset_index(drop=True)
)
print(f'After dedup: {len(df):,} unique (artist, album) pairs')

## 3. Match to MusicBrainz album_ids

In [ ]:
lookup = (
    pd.read_parquet(os.path.join(DATA_DIR, 'mb_album_artists.parquet'),
                    columns=['album_id', 'album_name', 'artist_name'])
    .drop_duplicates(subset='album_id')
    .reset_index(drop=True)
)
lookup['artist_norm'] = lookup['artist_name'].map(normalise)
lookup['album_norm']  = lookup['album_name'].map(normalise)

merged = df.merge(
    lookup[['album_id', 'artist_norm', 'album_norm']],
    on=['artist_norm', 'album_norm'],
    how='inner'
)

print(f'Last.fm rows         : {len(df):,}')
print(f'Matched to album_id  : {len(merged):,}  ({100*len(merged)/len(df):.1f}%)')
merged.head(3)

In [ ]:
# Sample unmatched rows — useful to spot naming issues
matched_keys = set(zip(merged['artist_norm'], merged['album_norm']))
unmatched = df[
    ~df.apply(lambda r: (r['artist_norm'], r['album_norm']) in matched_keys, axis=1)
].head(10)
print('Sample unmatched:')
print(unmatched[['artist_name', 'album_name']].to_string())

## 4. Min-max scale

In [ ]:
FEATURE_COLS = ['album_listeners', 'album_scrobbles', 'artist_listeners', 'artist_scrobbles']

def minmax(s):
    mn, mx = s.min(), s.max()
    return s * 0.0 if mx == mn else (s - mn) / (mx - mn)

for col in FEATURE_COLS:
    merged[col + '_scaled'] = minmax(merged[col])

merged[[c + '_scaled' for c in FEATURE_COLS]].describe().round(3)

## 5. Build sparse matrix

In [ ]:
with open(os.path.join(FEATURES_DIR, 'album_ids.pkl'), 'rb') as f:
    album_ids = np.asarray(pickle.load(f))
album_id_to_row = {int(a): i for i, a in enumerate(album_ids)}

mat = lil_matrix((len(album_ids), 4), dtype=np.float32)
hits = 0
for _, row in merged.iterrows():
    idx = album_id_to_row.get(int(row['album_id']))
    if idx is None: continue
    mat[idx, 0] = row['album_listeners_scaled']
    mat[idx, 1] = row['album_scrobbles_scaled']
    mat[idx, 2] = row['artist_listeners_scaled']
    mat[idx, 3] = row['artist_scrobbles_scaled']
    hits += 1

mat = mat.tocsr()
print(f'Albums with Last.fm data : {hits:,} / {len(album_ids):,}  ({100*hits/len(album_ids):.3f}%)')
print(f'Matrix shape             : {mat.shape},  nnz={mat.nnz:,}')

## 6. Save

In [ ]:
NPZ_PATH = os.path.join(FEATURES_DIR, 'album_lastfm_popularity_matrix.npz')
save_npz(NPZ_PATH, mat)
print(f'NPZ saved  : {NPZ_PATH}')

# Save matched table as parquet for reference
MATCHED_PATH = os.path.join(DATA_DIR, 'lastfm_album_matched.parquet')
merged.to_parquet(MATCHED_PATH, index=False)
print(f'Matched    : {MATCHED_PATH}')

## 7. Sanity check

In [ ]:
test   = load_npz(NPZ_PATH)
scores = np.asarray(test[:, 1].todense()).ravel()
top10  = np.argsort(-scores)[:10]
lk     = lookup.set_index('album_id')

print(f'Shape : {test.shape}')
print(f'NNZ   : {test.nnz:,}')
print('\nTop 10 by album scrobbles:')
for i, idx in enumerate(top10):
    aid = int(album_ids[idx])
    r   = lk.loc[aid] if aid in lk.index else {'album_name': '?', 'artist_name': '?'}
    print(f'  {i+1:2d}. {r["artist_name"]} -- {r["album_name"]}  ({scores[idx]:.4f})')